In [ ]:
import pandas as pd
from IPython.display import display, HTML
df = pd.read_csv('train.csv')

# 1. head, shape, info
display(HTML("<b>1. 5 Baris Pertama</b>")); display(df.head())
display(HTML(f"<b>2. Shape Dataset:</b> {df.shape[0]} baris, {df.shape[1]} kolom"))
display(HTML("<b>3. Info Dataset:</b>")); df.info()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

In [10]:
# 2. pembersihan data (missing values)
null_pct = (df.isnull().sum() / len(df)) * 100
missing_info = pd.DataFrame({'missing_count': df.isnull().sum(), 'percentage (%)': null_pct}).query('missing_count > 0').sort_values('percentage (%)', ascending=False)
display(HTML("<b>1. Atribut dengan Missing Value (Persentase)</b>")); display(missing_info)
# drop kolom > 50% missing
drop_cols = missing_info[missing_info['percentage (%)'] > 50].index.tolist()
df_clean = df.drop(columns=drop_cols).copy()
display(HTML(f"<b>2. Kolom Dihapus (>50% missing):</b> {drop_cols}"))
# imputasi tanpa warning (langsung assign df[col] = ...)
num_cols = df_clean.select_dtypes(include=['number']).columns
cat_cols = df_clean.select_dtypes(include=['object', 'category', 'string']).columns
for col in num_cols: df_clean[col] = df_clean[col].fillna(df_clean[col].median())
for col in cat_cols: df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
display(HTML(f"<b>3. Sisa Missing Value Setelah Imputasi:</b> {df_clean.isnull().sum().sum()}"))

,missing_count,percentage (%)
PoolQC,1453,99.520548
MiscFeature,1406,96.301370
Alley,1369,93.767123
Fence,1179,80.753425
MasVnrType,872,59.726027
FireplaceQu,690,47.260274
LotFrontage,259,17.739726
GarageType,81,5.547945
GarageYrBlt,81,5.547945
GarageFinish,81,5.547945


In [3]:
# Handling outlier pakai IQR Capping
for col in num_cols:
    if col not in ['Id', 'SalePrice']:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        df_clean[col] = np.where(
            df_clean[col] < lower, lower,
            np.where(df_clean[col] > upper, upper, df_clean[col])
        )

In [4]:
# One-Hot Encoding
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

# Scaling numerik
scaler = StandardScaler()
scale_cols = [c for c in num_cols if c not in ['Id', 'SalePrice']]
df_encoded[scale_cols] = scaler.fit_transform(df_encoded[scale_cols])

In [5]:
# Drop korelasi tinggi (> 0.85)
corr_matrix = df_encoded.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]
df_reduced = df_encoded.drop(columns=to_drop)

# PCA
X = df_reduced.drop(columns=['Id', 'SalePrice'])
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X)
print("Explained variance ratio (10 components):", sum(pca.explained_variance_ratio_))

Explained variance ratio (10 components): 0.636520416614494


In [6]:
output_csv = "5054251012_Catherine_Aprilia_Tugas_Praproses_Data.csv"
df_reduced.to_csv(output_csv, index=False)
print("Berhasil disimpan ke:", output_csv)

Berhasil disimpan ke: 5054251012_Catherine_Aprilia_Tugas_Praproses_Data.csv
